In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sales = pd.read_csv("../Data/Cleaned/sales_transactions_clean.csv")
customers = pd.read_csv("../Data/Cleaned/customers_clean.csv")
products = pd.read_csv("../Data/Cleaned/products_clean.csv")
inventory = pd.read_csv("../Data/Cleaned/inventory_clean.csv")
suppliers = pd.read_csv("../Data/Cleaned/suppliers_clean.csv")

In [3]:
sales_fe = sales.copy()

print(sales_fe.shape)

(190000, 15)


In [ ]:
#Convert Order Date
sales_fe["order_date"] = pd.to_datetime(sales_fe["order_date"])

sales_fe["order_date"].head()

0   2023-04-15
1   2024-01-14
2   2024-10-20
3   2020-01-04
4   2024-03-17
Name: order_date, dtype: datetime64[ns]

In [ ]:
#Date Features
sales_fe["year"] = sales_fe["order_date"].dt.year
sales_fe["quarter"] = sales_fe["order_date"].dt.quarter
sales_fe["month"] = sales_fe["order_date"].dt.month
sales_fe["month_name"] = sales_fe["order_date"].dt.month_name()
sales_fe["day"] = sales_fe["order_date"].dt.day
sales_fe["day_name"] = sales_fe["order_date"].dt.day_name()
sales_fe["week"] = sales_fe["order_date"].dt.isocalendar().week

sales_fe.head()

,order_id,order_date,customer_id,product_id,supplier_id,city,channel,quantity,unit_price,discount_pct,...,returned,return_reason,payment_method,year,quarter,month,month_name,day,day_name,week
0,ORD000001,2023-04-15,CUS024281,MED000014,SUP000211,Mumbai,Retail Pharmacy,3,505.20,12.15,...,No,NaN,UPI,2023,2,4,April,15,Saturday,15
1,ORD000002,2024-01-14,CUS023216,MED000010,SUP000086,Hyderabad,Retail Pharmacy,17,2724.52,6.84,...,No,NaN,Bank Transfer,2024,1,1,January,14,Sunday,2
2,ORD000003,2024-10-20,CUS007158,MED000016,SUP000083,Mumbai,hospital,11,1135.24,3.72,...,No,NaN,UPI,2024,4,10,October,20,Sunday,42
3,ORD000004,2020-01-04,CUS021793,MED000512,SUP000107,Mumbai,Hospital,51,392.29,0.00,...,No,NaN,UPI,2020,1,1,January,4,Saturday,1
4,ORD000005,2024-03-17,CUS011808,MED001444,SUP000464,Delhi,Retail Pharmacy,4,3586.72,0.00,...,No,NaN,UPI,2024,1,3,March,17,Sunday,11


In [ ]:
#Gross Sales
sales_fe["gross_sales"] = (
    sales_fe["quantity"] *
    sales_fe["unit_price"]
)

sales_fe["gross_sales"].describe()

count    1.900000e+05
mean     1.015256e+05
std      1.664368e+05
min      3.129000e+01
25%      1.723287e+04
50%      4.756600e+04
75%      1.165367e+05
max      3.193584e+06
Name: gross_sales, dtype: float64

In [ ]:
#Discount Amount
sales_fe["discount_amount"] = (
    sales_fe["gross_sales"] *
    sales_fe["discount_pct"] / 100
)

sales_fe["discount_amount"].describe()

count    1.900000e+05
mean     1.287927e+04
std      2.802632e+04
min      0.000000e+00
25%      1.190377e+03
50%      4.491815e+03
75%      1.328519e+04
max      1.568338e+06
Name: discount_amount, dtype: float64

In [ ]:
#Profit Margin %
sales_fe["profit_margin_pct"] = (
    sales_fe["profit"] /
    sales_fe["revenue"]
) * 100

sales_fe["profit_margin_pct"] = sales_fe["profit_margin_pct"].round(2)

sales_fe["profit_margin_pct"].describe()

count    190000.00000
mean         28.40201
std          27.71113
min       -3325.24000
25%          15.22000
50%          32.42000
75%          43.64000
max        2102.92000
Name: profit_margin_pct, dtype: float64

In [ ]:
#Revenue Per Unit
sales_fe["revenue_per_unit"] = (
    sales_fe["revenue"] /
    sales_fe["quantity"]
)

sales_fe["revenue_per_unit"].round(2)

sales_fe["revenue_per_unit"].describe()

count    190000.000000
mean       2730.212180
std        1832.603422
min          13.252000
25%        1016.068375
50%        2567.331000
75%        4050.566284
max        8266.460000
Name: revenue_per_unit, dtype: float64

In [ ]:
#Profit Per Unit
sales_fe["profit_per_unit"] = (
    sales_fe["profit"] /
    sales_fe["quantity"]
)

sales_fe["profit_per_unit"] = sales_fe["profit_per_unit"].round(2)

sales_fe["profit_per_unit"].describe()

count    190000.000000
mean        918.877254
std         948.774356
min       -1998.340000
25%         138.887500
50%         621.055000
75%        1499.575000
max        4898.630000
Name: profit_per_unit, dtype: float64

In [11]:
#Return Flag
sales_fe["return_flag"] = sales_fe["returned"].map(
    {
        "Yes":1,
        "No":0
    }
)

sales_fe["return_flag"].value_counts()

return_flag
0.0    177701
1.0      4134
Name: count, dtype: int64

In [12]:
#High Discount Orders
sales_fe["high_discount"] = (
    sales_fe["discount_pct"] >= 20
)

sales_fe["high_discount"].value_counts()

high_discount
False    159014
True      30986
Name: count, dtype: int64

In [13]:
#Loss Making Orders
sales_fe["loss_order"] = (
    sales_fe["profit"] < 0
)

sales_fe["loss_order"].value_counts()

loss_order
False    170566
True      19434
Name: count, dtype: int64

In [14]:
#Revenue Category
sales_fe["revenue_category"] = pd.qcut(
    sales_fe["revenue"],
    q=3,
    labels=[
        "Low",
        "Medium",
        "High"
    ]
)

sales_fe["revenue_category"].value_counts()

revenue_category
High      63334
Low       63333
Medium    63333
Name: count, dtype: int64

In [15]:
#Order Size
sales_fe["order_size"] = pd.cut(
    sales_fe["quantity"],
    bins=[0,20,50,100,400],
    labels=[
        "Small",
        "Medium",
        "Large",
        "Bulk"
    ]
)

sales_fe["order_size"].value_counts()

order_size
Small     98050
Medium    58425
Large     23375
Bulk      10150
Name: count, dtype: int64

In [16]:
#Dataset Information
sales_fe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 190000 entries, 0 to 189999
Data columns (total 32 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   order_id           190000 non-null  object        
 1   order_date         190000 non-null  datetime64[ns]
 2   customer_id        190000 non-null  object        
 3   product_id         190000 non-null  object        
 4   supplier_id        184829 non-null  object        
 5   city               186000 non-null  object        
 6   channel            190000 non-null  object        
 7   quantity           190000 non-null  int64         
 8   unit_price         190000 non-null  float64       
 9   discount_pct       190000 non-null  float64       
 10  revenue            190000 non-null  float64       
 11  profit             190000 non-null  float64       
 12  returned           190000 non-null  object        
 13  return_reason      12016 non-null   object  

In [18]:
#Save Feature Engineered Dataset
sales_fe.to_csv(
    "../Data/Feature Engineered/sales_feature_engineered.csv",
    index=False
)

print("Feature Engineering Completed Successfully!")

Feature Engineering Completed Successfully!
